In [ ]:
import hashlib, importlib.util, pathlib, urllib.request
from google.colab import files, runtime

RUNNER_URL = 'https://raw.githubusercontent.com/lluiseriksson/THE-ERIKSSON-PROGRAMME/0eefbd484993e06293d20032668a61409e72a13d/scripts/colab_c6d_step3_localized_precision_validation.py'
RUNNER_SHA256 = '1f2791a43bd7db5a87fa1336b92167014b204a2777394b1fb13c6ac4265b79d6'
RUNNER_FILE = pathlib.Path("/content/c6d_step3_v3_runner.py")
with urllib.request.urlopen(RUNNER_URL) as response:
    runner_source = response.read()
measured = hashlib.sha256(runner_source).hexdigest()
print("RUNNER_TRANSPORT_SHA256=" + measured, flush=True)
if measured != RUNNER_SHA256:
    raise RuntimeError("RUNNER_TRANSPORT_HASH_MISMATCH")
RUNNER_FILE.write_bytes(runner_source)
spec = importlib.util.spec_from_file_location("c6d_step3_v3_runner", RUNNER_FILE)
if spec is None or spec.loader is None:
    raise RuntimeError("C6D_DIAGNOSTIC_RUNNER_IMPORT_FAILED")
module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(module)
if module.runner.SOURCE_SHA != '557a472e96509d3473b925cb07114292fc28587c':
    raise RuntimeError("C6D_DIAGNOSTIC_SOURCE_SHA_MISMATCH")
if module.runner.RUNNER_REV != 'c6d-step3-localized-precision-v4':
    raise RuntimeError("C6D_DIAGNOSTIC_RUNNER_REV_MISMATCH")
if len(module.runner.QUEUE) != 9:
    raise RuntimeError("C6D_DIAGNOSTIC_ORIGINAL_QUEUE_MISMATCH")
if module.runner.QUEUE[0][0] != "00_c6d_step3_clm_extensionality_repro":
    raise RuntimeError("C6D_DIAGNOSTIC_FIRST_STAGE_MISMATCH")
module.runner.RUNNER_REV = 'c6d-step3-clm-repro-diagnostic-v2'
module.runner.ROOT = pathlib.Path("/content/hrpoly-c6d-step3-clm-diagnostic")
module.runner.EVIDENCE = pathlib.Path("/content/hrpoly-c6d-step3-clm-diagnostic-evidence")
module.runner.ARCHIVE = pathlib.Path("/content/hrpoly-c6d-step3-clm-diagnostic-evidence.tar.gz")
module.runner.PATH_MANIFEST = pathlib.Path("/content/hrpoly-c6d-step3-clm-diagnostic-paths.txt")
module.runner.QUEUE = module.runner.QUEUE[:1]
runtime.unassign = lambda: print("RUNTIME_UNASSIGN_DEFERRED=1", flush=True)
diagnostic_exit = module.runner.main()
files.download(str(module.runner.ARCHIVE))
print("DIAGNOSTIC_EXIT=" + str(diagnostic_exit), flush=True)
print("RUNTIME_RETAINED_FOR_EVIDENCE=1", flush=True)
if diagnostic_exit != 0:
    raise RuntimeError("C6D_DIAGNOSTIC_FAILED_AS_EXPECTED")
